[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/liquid_crystals/liquid_crystals.ipynb)

# Nematic Liquid Crystals

This notebook computes the torque restoring molecular alignment in a liquid-crystal film and the motion of a pair of defects. The molecules prefer parallel axes, with neither end distinguished. Around a half-charge defect their axes turn halfway around; the alignment weakens at its core, allowing opposite defects to disappear together.

The alignment is modelled as a symmetric traceless map on directions. Its change under a bivector turn gives the free energy's orientational derivative, a form on turns. The restoring couple represents that form through `couple & turn`. A virtual turn shows its predicted energy change before the film relaxes; active stress then drives the same film on a substrate.

In the tensor notation of nematodynamics the alignment is $Q=S(\mathbf n\mathbf n^T-I/2)$, unchanged by $\mathbf n\mapsto-\mathbf n$. The [active-defect mechanism](https://arxiv.org/abs/2209.00333) motivates the driven comparison; the film here uses a compressible local-drag model.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import SparseExtensor, spdiag
from examples.util import shapes
from examples.util.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.fluids.liquid_crystals import render

ga = Algebra("x+y+")
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                      # Scalar+
Point = ga.gatype.antivector()                                   # Antivector-
Plane = ga.gatype.vector()                                       # Vector+
Turn = ga.gatype.bivector()                                      # Bivector+
Couple = ga.gatype.antibivector()                                # Antibivector-
Order = ga.gatype((Point, Point))                                # Point- <- Point-
Slope = ga.gatype((Scalar, Turn))                                # Scalar+ <- Turn+
work = Couple & Turn                                            # [] Scalar+ <- (Couple-, Turn+)

## Molecular alignment

An aligned patch stretches a direction along the molecular axis and shortens its perpendicular by the same amount. Reversing the axis leaves that map unchanged. A defect therefore needs no cut across which a signed director must jump; the alignment itself is continuous, shrinking towards zero at the core.

In [ ]:
half_width, n_side = 6.0, 49
separation, core_radius = 2.4, 0.45
axis = np.linspace(-half_width, half_width, n_side)
topology, coordinates = shapes.box([axis, axis])
vertices = mv.vector(coordinates).dual().field()                   # Point-[vertices]
centres = mv.vector([[-separation / 2 + 0.03, 0.07],
                     [separation / 2 + 0.03, 0.07]]).dual()       # [defects] Point-

In [ ]:
def alignment(directors: Point, strength: Scalar) -> Order:
    """The alignment map of these unit molecular axes, with this degree of order."""
    return strength * (directors * (directors.dual_inverse() & Point) - Point / 2)


def defect_pair(vertices: Point, centres: Point, core_radius: float, bearing: Turn) -> Order:
    """Opposite half-charge defects, with a smooth loss of order at their cores."""
    positive, negative = vertices - centres[0], vertices - centres[1]
    # The relative radial turn winds once at one core and backwards at the other. Its quarter power turns a
    # molecular axis through half that angle; crossing the logarithm's cut only reverses the axis.
    relative = positive.normalized().dual_inverse() * negative.normalized().dual_inverse()
    turn = (relative.log() / 4 + bearing * -0.5).exp()
    directors = turn >> mv.x.dual()                              # Point-[vertices]
    radii = positive.norm() * negative.norm()
    strength = radii / ((positive.scalar_norm_squared() + core_radius ** 2) *
                        (negative.scalar_norm_squared() + core_radius ** 2)).square_root()
    return alignment(directors, strength)

initial = defect_pair(vertices, centres, core_radius, mv.xy * 0)
save_figure(render.draw_setup(topology, vertices, initial), "liquid_crystals_setup");

The fixed grain stretches symmetrically along the molecular axes and shortens as order weakens. Its bending shows the alignment turning, with neither end of an axis preferred. The blue tint marks the loss of order. Copper and blue dots mark the opposite half-charge defects, located by the winding around a mesh triangle.

In [ ]:
def windings(topology: Topology, order: Order, reference: Point) -> Turn:
    """Each triangle's director winding, in whole turns, represented in the film's turning plane."""
    # A traceless alignment acting on a fixed direction turns through twice the molecular angle. Its edge
    # logarithms therefore count half-charge defects without choosing a sign for any molecular axis.
    ends = topology.incidence[1, 0] * order(reference)[..., None]
    tails, heads = ends[..., 0], ends[..., 1]
    turns = (heads.dual_inverse() * tails.dual_inverse()).normalized().log()
    return (topology.d[1] * turns) / (4 * np.pi)

## Free energy

The bulk energy favours strong alignment, while the elastic energy makes neighbouring patches resist different alignments. Their competition gives a defect its finite core. The molecular field is the negative energy slope per unit area, within symmetric traceless maps; it can change both the strength and the direction of alignment.

In tensor notation the energy is $F=\int [a(\mathrm{tr}(Q^2)-1/2)^2+K|\nabla Q|^2]/2\,dA$, and $H=-\delta F/\delta Q$.

In [ ]:
elastic, bulk = 0.6, 3.0
corners = topology.incidence[2, 0]                                 # [corners] Scalar+[faces <- vertices]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)       # [corners] Point-[faces]
volume = sides[0] & sides[1]                                      # Scalar-[faces]
gradients = sides / volume                                       # [corners] Plane+[faces]
areas = volume.norm() / 2                                         # Scalar+[faces]
cells = (~corners * (areas / 3)).sum(axis=0)                      # Scalar+[vertices]
# Isotropic elasticity pairs each corner gradient with the dual of the other.
local = areas * (gradients[:, None] & gradients[None, :].dual())
stiffness = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)

In [ ]:
def free_energy(stiffness: SparseExtensor, cells: Scalar, elastic: float, bulk: float, order: Order) -> Scalar:
    """The film's free energy: a bulk preference for unit order and the elastic cost of spatial variation."""
    # This is the trace of each local map, leaving the spatial sites uncontracted.
    squared_order = order(order).batch().trace().field()          # Scalar+[vertices]
    well = bulk * (squared_order - 0.5) ** 2 / 2                # Scalar+[vertices]
    distortion = order(stiffness * order).batch().trace().field() # Scalar+[vertices]
    return (cells * well + elastic * distortion / 2).sites.sum()


def molecular_field(stiffness: SparseExtensor, cells: Scalar, elastic: float, bulk: float, order: Order) -> Order:
    """The negative free-energy derivative per unit area, within symmetric traceless maps."""
    squared_order = order(order).batch().trace().field()          # Scalar+[vertices]
    return -2 * bulk * (squared_order - 0.5) * order - elastic * (stiffness * order) / cells

## The derivative with respect to a bivector turn

A turn moves both the directions carried by an alignment map and the directions it reads. Contracting that change with the energy slope gives a form on the bivector turn. Its negative is the work of the restoring couple. A rotation leaves the degree of order fixed: this derivative describes orientation, while the full molecular field also allows a core to melt.

In [ ]:
def turn_slope(order: Order, molecular: Order) -> Slope:
    """The free-energy density's first derivative with respect to a bivector turn of the alignment."""
    # Both the direction carried by the map and the direction it reads turn with the molecules.
    tangent = order.commutator(Turn) - order(Point.commutator(Turn))
    # The trace pairs the molecular field with the map's change; the molecular field is minus the slope.
    return -molecular(tangent).batch().trace(0, 1).field()

In [ ]:
probe_width, imposed_turn, turn_span, n_turn_frames = 0.8, 1.0, 0.25, 61
frame_duration_ms = 40
# A smooth patch near one defect is turned out of alignment with its neighbours.
offset = vertices - (centres[0] + mv.y.dual() * probe_width)
probe = mv.xy * (-offset.scalar_norm_squared() / (2 * probe_width ** 2)).exp()  # Turn+[vertices]
imprint = (-probe * imposed_turn / 2).exp()
strained = imprint >> initial(imprint << Point)
molecular = molecular_field(stiffness, cells, elastic, bulk, strained)
slope = turn_slope(strained, molecular)                            # (Scalar+ <- Turn+)[vertices]
couple = work.solve(-slope)                                       # Couple-[vertices]
# The couple's work is minus the first change of free energy.
predicted_slope = -(cells * (couple & probe)).sites.sum()          # [] Scalar+
angles = np.linspace(-turn_span, turn_span, n_turn_frames)

The close view isolates the patch above the blue defect, where the imposed turn is varied about this strained state. The following cell takes a few seconds, mostly drawing the animation.

In [ ]:
def virtual_turns() -> Iterator[Order]:
    for angle in angles:
        rotor = (-probe * angle / 2).exp()
        yield (rotor >> strained(rotor << Point))[None]


turn_states = show_states(virtual_turns(), render.animate, topology, vertices, render.turn_camera,
                         name="liquid_crystals_turn", duration_ms=frame_duration_ms, colors=render.n_colours)

In [ ]:
turn_energies = stack([free_energy(stiffness, cells, elastic, bulk, state[0]) for state in turn_states])
baseline = free_energy(stiffness, cells, elastic, bulk, strained)
save_figure(render.draw_derivative(angles, turn_energies, baseline, predicted_slope), "liquid_crystals_derivative");

The blue curve is the energy change under the local turn. The dashed copper line is its tangent, predicted by the couple's work. Only a first derivative is used; its argument is a field of bivectors.

## Relaxation by the bivector derivative

Rotational friction converts the restoring couple into an angular velocity. The rotor generated by that velocity turns the alignment, preserving its strength at every site. This run follows the bivector derivative alone; the weakened cores remain part of its prescribed order profile. The following two cells take a few seconds, mostly drawing.

In [ ]:
def turn_relaxation(topology: Topology, stiffness: SparseExtensor, cells: Scalar, elastic: float, bulk: float,
                    initial: Order, time_step: float, n_frames: int, n_substeps: int) -> Iterator[Order]:
    """The alignment rotating down its energy slope, with its degree of order fixed at every site."""
    # Unit rotational friction is a positive form on bivectors. Solving it against the negative derivative
    # gives the angular velocity whose dissipated power is the couple's work.
    friction = -Turn.scalar_product(Turn)                        # [] Scalar+ <- (Turn+, Turn+)
    boundary = topology.reading(topology.exterior[0], 0)

    def step(order: Order, _) -> tuple[Order, None]:
        molecular = molecular_field(stiffness, cells, elastic, bulk, order)
        angular_velocity = friction.solve(-turn_slope(order, molecular))
        angular_velocity = angular_velocity - ~boundary * (boundary * angular_velocity)
        rotor = (-angular_velocity * time_step / 2).exp()
        return rotor >> order(rotor << Point), None

    @jax.jit
    def advance(order: Order) -> Order:
        return jax.lax.scan(step, order, None, length=n_substeps)[0]

    order = initial
    yield order
    for _ in range(n_frames - 1):
        order = advance(order)
        yield order

In [ ]:
rotation_step, n_relaxation_frames, n_relaxation_substeps = 0.0075, 61, 2
rotation_states = show_states(turn_relaxation(topology, stiffness, cells, elastic, bulk, strained[None],
                                             rotation_step, n_relaxation_frames, n_relaxation_substeps),
                             render.animate, topology, vertices, render.turn_camera,
                             name="liquid_crystals_torque", duration_ms=frame_duration_ms, colors=render.n_colours)

In [ ]:
rotation_energies = stack([free_energy(stiffness, cells, elastic, bulk, state) for state in rotation_states])
rotation_times = np.arange(n_relaxation_frames) * rotation_step * n_relaxation_substeps
save_figure(render.draw_release(rotation_times, rotation_energies[:, 0]), "liquid_crystals_torque_energy");

The grain bends as the axes turn under the restoring torque, while the blue patches of reduced order stay fixed. The curve shows the free energy released as this orientational distortion relaxes.

## Active stress and defect motion

An active patch pushes or pulls along its molecular alignment. Where that alignment varies, the stress has a net force, balanced here by local drag against the substrate. The resulting velocity transports and turns the molecules. The local drag closure allows compression. Activity supplies energy, so the driven film need not descend its free energy.

In tensor notation this model reads $\mathbf u=-\zeta\,\nabla\cdot Q$, $\partial_t Q=H-\mathbf u\cdot\nabla Q+\Omega Q-Q\Omega$, with $\Omega=(\nabla\mathbf u-(\nabla\mathbf u)^T)/2$. The flow-alignment parameter is zero.

In [ ]:
def active_velocity(topology: Topology, areas: Scalar, gradients: Plane, cells: Scalar,
                    activity: np.ndarray, order: Order) -> Point:
    """The velocity driven by minus the alignment's divergence, at unit substrate drag."""
    corners = topology.incidence[2, 0]
    face_order = corners * order[..., None]                     # [..., corners] (Point- <- Point-)[faces]
    # Isotropic substrate drag turns the stress divergence into a velocity along its force.
    divergence = face_order(gradients.dual()).sum(axis=-1)     # [...] Point-[faces]
    velocity = -activity * ((~corners * (areas * divergence / 3)[..., None]).sum(axis=-1)) / cells
    boundary = topology.reading(topology.exterior[0], 0)
    return velocity - ~boundary * (boundary * velocity)

In [ ]:
def rate(topology: Topology, areas: Scalar, gradients: Plane, cells: Scalar, stiffness: SparseExtensor,
         elastic: float, bulk: float, activity: np.ndarray, order: Order) -> Order:
    """Relaxation, advection and corotation of the film's alignment."""
    molecular = molecular_field(stiffness, cells, elastic, bulk, order)
    velocity = active_velocity(topology, areas, gradients, cells, activity, order)
    corners = topology.incidence[2, 0]
    local_order = corners * order[..., None]
    local_velocity = corners * velocity[..., None]
    face_velocity = local_velocity.mean(axis=-1)                # [...] Point-[faces]
    advection = (local_order * (gradients & face_velocity[..., None])).sum(axis=-1)
    velocity_gradient = (local_velocity * (gradients & Point)).sum(axis=-1)
    # Half the curl is the plane and rate of the fluid's rotation.
    spin = Point.reverse().commutator(velocity_gradient(Point)).batch().contract(1, 2).field() / 2
    face_order = local_order.mean(axis=-1)
    corotation = face_order.commutator(spin) - face_order(Point.commutator(spin))
    transported = (~corners * (areas * (corotation - advection) / 3)[..., None]).sum(axis=-1) / cells
    changing = molecular + transported
    boundary = topology.reading(topology.exterior[0], 0)
    return changing - ~boundary * (boundary * changing)

The boundary holds its initial alignment. The passive film and the films with opposite activity start from the same defect pair; each is a batch member, while the mesh vertices remain a field.

In [ ]:
activities = np.array([0.0, 0.4, -0.4])
time_step, n_substeps, n_frames = 0.015, 2, 91
start = initial[None].broadcast_to((len(activities),))

In [ ]:
def evolution(topology: Topology, areas: Scalar, gradients: Plane, cells: Scalar, stiffness: SparseExtensor,
              elastic: float, bulk: float, activity: np.ndarray, initial: Order, time_step: float,
              n_frames: int, n_substeps: int) -> Iterator[Order]:
    """The film's alignment at each frame, advanced with a midpoint step."""
    def step(order: Order, _) -> tuple[Order, None]:
        halfway = order + time_step / 2 * rate(topology, areas, gradients, cells, stiffness,
                                               elastic, bulk, activity, order)
        following = order + time_step * rate(topology, areas, gradients, cells, stiffness,
                                              elastic, bulk, activity, halfway)
        return following, None

    @jax.jit
    def advance(order: Order) -> Order:
        return jax.lax.scan(step, order, None, length=n_substeps)[0]

    order = initial
    yield order
    for _ in range(n_frames - 1):
        order = advance(order)
        yield order

The next cell follows all films together and takes about ten seconds, mostly drawing.

In [ ]:
states = show_states(evolution(topology, areas, gradients, cells, stiffness, elastic, bulk, activities,
                              start, time_step, n_frames, n_substeps),
                     render.animate, topology, vertices, render.defect_camera,
                     name="liquid_crystals_defects", duration_ms=frame_duration_ms, colors=render.n_colours)

The passive film is at the top; positive and negative activity are in the middle and at the bottom. All rows share one fixed window and scale, cropped around the defects and their travel. The passive pair annihilates. Active stress changes the pair's motion and the time it takes to disappear; reversing it changes that motion again.

In [ ]:
energies = stack([free_energy(stiffness, cells, elastic, bulk, state) for state in states])
times = np.arange(n_frames) * time_step * n_substeps
save_figure(render.draw_energy(times, energies), "liquid_crystals_energy");

The blue passive curve decreases throughout. Copper and green correspond to the middle and bottom films. Their different paths reflect the work of the active stress as well as the dissipation.

## A crowded film

Many half-charge defects move and meet at once in a disordered film. Nearby pairs can disappear while larger regions keep turning and rearranging. Active stress drives the film below, whose alignment vanishes smoothly at each defect core.

In [ ]:
def disordered_alignment(vertices: Point, wavevectors: Plane, phases: np.ndarray,
                         amplitudes: Order, softness: float) -> Order:
    """A smooth, disordered alignment, with half-charge defects where the wave sum vanishes."""
    waves = (wavevectors & vertices[..., None]) + phases
    order = (amplitudes * waves.cos()).sum(axis=-1)
    # Saturation gives strongly aligned domains while leaving each zero a continuous, disordered core.
    strength_squared = 2 * order(order).batch().trace().field()
    return order / (strength_squared + softness ** 2).square_root()

In [ ]:
dense_half_width, n_dense_side = 8.0, 65
n_waves, wave_number, softness, seed = 32, 1.8, 0.2, 7
dense_activities = np.array([1.2])
n_dense_substeps = 2
rng = np.random.default_rng(seed)
dense_axis = np.linspace(-dense_half_width, dense_half_width, n_dense_side)
dense_topology, dense_coordinates = shapes.box([dense_axis, dense_axis])
dense_vertices = mv.vector(dense_coordinates).dual().field()
dense_wavevectors = mv.vector(rng.normal(size=(n_waves, 2)) * wave_number)
dense_phases = rng.uniform(0, 2 * np.pi, n_waves)
dense_bearings = mv.xy * rng.uniform(0, np.pi, n_waves)
dense_directors = (-dense_bearings / 2).exp() >> mv.x.dual()
dense_amplitudes = alignment(dense_directors, 1 / np.sqrt(n_waves))
dense_initial = disordered_alignment(dense_vertices, dense_wavevectors, dense_phases, dense_amplitudes, softness)
save_figure(render.draw_dense_setup(dense_topology, dense_vertices, dense_initial), "liquid_crystals_dense_setup");

The same free energy and local drag act on this film. Its boundary holds the initial alignment. The next cells take about half a minute, mainly drawing the animation.

In [ ]:
dense_corners = dense_topology.incidence[2, 0]
dense_sides = dense_topology.boundary[2, 1] * (dense_topology.d[0] * dense_vertices)
dense_volume = dense_sides[0] & dense_sides[1]
dense_gradients = dense_sides / dense_volume
dense_areas = dense_volume.norm() / 2
dense_cells = (~dense_corners * (dense_areas / 3)).sum(axis=0)
dense_local = dense_areas * (dense_gradients[:, None] & dense_gradients[None, :].dual())
dense_stiffness = (~dense_corners[:, None] * spdiag(dense_local) * dense_corners[None, :]).sum(axis=0).sum(axis=0)
dense_start = dense_initial[None].broadcast_to((len(dense_activities),))

In [ ]:
dense_states = show_states(evolution(dense_topology, dense_areas, dense_gradients, dense_cells, dense_stiffness,
                                    elastic, bulk, dense_activities, dense_start, time_step, n_frames, n_dense_substeps),
                           render.animate, dense_topology, dense_vertices, render.dense_camera,
                           name="liquid_crystals_dense", duration_ms=40, colors=render.n_colours)

Copper and blue mark opposite half charges. Close pairs disappear first; the surviving defects and the grain around them continue moving. Active stress changes their paths and the way neighbouring regions align.

In [ ]:
dense_energies = stack([free_energy(dense_stiffness, dense_cells, elastic, bulk, state) for state in dense_states])
dense_times = np.arange(n_frames) * time_step * n_dense_substeps
save_figure(render.draw_energy(dense_times, dense_energies), "liquid_crystals_dense_energy");

In [ ]:
# checks
# The two initial defects have opposite half charges.
charges = windings(topology, initial, mv.x.dual()).scalar_product(~mv.xy).to_array()
np.testing.assert_allclose(np.sort(charges[np.abs(charges) > 0.1]), [-0.5, 0.5], atol=1e-8)
# Passive motion dissipates free energy and removes the pair.
assert np.all(np.diff(energies[:, 0].to_array()) <= 0)
assert np.all(windings(topology, states[-1][0], mv.x.dual()).norm() < 0.01)
# Evolution preserves the symmetry and trace of the alignment.
last = states[-1].batch()
np.testing.assert_allclose(last.trace().kernel, 0, atol=1e-8)
np.testing.assert_allclose((last - last.adjoint()).kernel, 0, atol=1e-8)
# Rotation driven by the bivector derivative dissipates energy without changing the degree of order.
assert np.all(np.diff(rotation_energies[:, 0].to_array()) <= 0)
rotation_first, rotation_last = rotation_states[0].batch(), rotation_states[-1].batch()
np.testing.assert_allclose(rotation_first(rotation_first).trace().kernel,
                           rotation_last(rotation_last).trace().kernel, atol=1e-8)
# The crowded active film retains finite, symmetric traceless alignment.
assert np.all(dense_energies.isfinite())
dense_last = dense_states[-1].batch()
np.testing.assert_allclose(dense_last.trace().kernel, 0, atol=1e-8)
np.testing.assert_allclose((dense_last - dense_last.adjoint()).kernel, 0, atol=1e-8)